In [2]:
!pip uninstall google-generativeai -y
!pip install google-genai tavily-python python-dotenv --upgrade

Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Successfully uninstalled google-generativeai-0.8.6
  Using cached websockets-15.0.1-cp310-cp310-win_amd64.whl.metadata (7.0 kB)
   ---------------------------------------- 0.0/724.7 kB ? eta -:--:--
   ---------------------------------------- 724.7/724.7 kB 5.0 MB/s  0:00:00
Using cached websockets-15.0.1-cp310-cp310-win_amd64.whl (176 kB)

   ---------------------------------------- 0/3 [websockets]
   -------------------------- ------------- 2/3 [google-genai]
   -------------------------- ------------- 2/3 [google-genai]
   -------------------------- ------------- 2/3 [google-genai]
   -------------------------- ------------- 2/3 [google-genai]
   -------------------------- ------------- 2/3 [google-genai]
   -------------------------- ------------- 2/3 [google-genai]
   -------------------------- ------------- 2/3 [google-genai]
   -------------------------- ------------- 2/3 [google-ge

In [5]:
import os
import json
from tavily import TavilyClient
from dotenv import load_dotenv
from IPython.display import Markdown, display
from google import genai

In [6]:
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

print("Gemini:", gemini_api_key[:5])
print("Tavily:", tavily_api_key[:5])

client = genai.Client(api_key=gemini_api_key)
tavily = TavilyClient(api_key=tavily_api_key)

Gemini: AIzaS
Tavily: tvly-


In [11]:
def tavily_search(query):
    response = tavily.search(query=query, max_results=2)
    return response["results"]

In [8]:
tool_instructions = """
You are an AI research assistant.

You have access to a search tool called "tavily_search".

If the question requires up-to-date information,
respond ONLY in this JSON format:

{
  "action": "search",
  "query": "search query here"
}

If no search is needed,
respond normally in plain text.
"""

In [12]:
def run_agent(user_query):
    
    initial_prompt = tool_instructions + "\n\nUser Question: " + user_query
    
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=initial_prompt
    )
    
    text = response.text
    
    try:
        parsed = json.loads(text)
        
        if parsed.get("action") == "search":
            
            search_results = tavily_search(parsed["query"])
            
            follow_up_prompt = f"""
            You searched Tavily and got the following results:

            {search_results}

            Based on this information, answer the user's question:
            {user_query}
            """
            
            final_response = client.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=follow_up_prompt
            )
            
            return final_response.text
    
    except:
        return text

In [14]:
question = "What are people saying about the new GPT-5 Model?"

display(Markdown("### 🔎 User Question"))
display(Markdown(question))

answer = run_agent(question)

display(Markdown("### 🤖 Agent Response"))
display(Markdown(answer))

### 🔎 User Question

What are people saying about the new GPT-5 Model?

### 🤖 Agent Response

Based on the search results, here's what people are saying about the new GPT-5 model:

*   **Rumors suggest OpenAI has built GPT-5 but is keeping it internal.** The primary reason cited is that the return on investment (ROI) is greater for OpenAI by *not* releasing it to ChatGPT users. This ROI is not necessarily monetary.
*   **Similar strategies to Anthropic:** It's speculated that OpenAI is following a similar path to Anthropic with their Claude 3.5 Opus. Anthropic reportedly used its Opus 3.5 to generate synthetic data and for reward modeling to improve their Sonnet model, rather than releasing Opus 3.5 directly due to underwhelming results relative to its cost.
*   **Distillation as a key technique:** The implication is that OpenAI is using distillation, a process where a smaller model is trained to mimic a larger one, to improve their existing models (like those used in ChatGPT) using data generated or insights gained from GPT-5. This is seen as a way to control costs and achieve better results with more accessible models.

In essence, the prevailing rumor is that GPT-5 exists, but OpenAI is strategically using its capabilities internally to enhance their current offerings rather than a direct public release, mirroring strategies seen with Anthropic's Claude 3.5 Opus.

In [26]:
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv
from IPython.display import Markdown, display

In [27]:
load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [28]:
from google.genai import types

In [29]:
analyst_instructions = """
You are a world-class market research assistant.

You have access to:
- Real-time web search
- Python code execution

When needed:
- Use web search for current prices or market data.
- Use code execution for simulations and calculations.
- Integrate results clearly.
- When using web search, cite findings.
- When using code, explain the results clearly.
"""

In [30]:
query = """
Find the current price of a Tesla Cybertruck in Canada.
Then simulate how the price would change if import tariffs increased
from 5% to 20% in 1% increments.
For each tariff rate, calculate the new price using Python
and provide a summary of the results.
"""

In [36]:
tools = [
    types.Tool(google_search={}),
    types.Tool(code_execution={})
]

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=analyst_instructions + "\n\nUser Question:\n" + query,
    config=types.GenerateContentConfig(tools=tools)
)

display(Markdown("### 🤖 Analyst Agent Response"))
display(Markdown(response.text))

### 🤖 Analyst Agent Response

The current price of a Tesla Cybertruck in Canada (dual-motor AWD) is reported to be $114,990 CAD, as of February 2025. This is the Manufacturer's Suggested Retail Price (MSRP) and does not include freight, PDI, dealer fees, or applicable taxes such as the federal luxury tax.

Now, let's simulate how the price would change if import tariffs increased from 5% to 20% in 1% increments. We will use the base price of $114,990 CAD for this simulation.

The current base price of a Tesla Cybertruck (dual-motor AWD) in Canada is $114,990 CAD, as reported in February 2025. This price does not include additional fees, PDI, or taxes.

Here's a summary of how the price would change with increasing import tariffs, based on the $114,990 CAD base price:

**Simulation Results (Base Price: $114,990 CAD)**

*   **Tariff Rate: 5%** -> New Price: **$120,739.50 CAD**
*   **Tariff Rate: 6%** -> New Price: **$121,889.40 CAD**
*   **Tariff Rate: 7%** -> New Price: **$123,039.30 CAD**
*   **Tariff Rate: 8%** -> New Price: **$124,189.20 CAD**
*   **Tariff Rate: 9%** -> New Price: **$125,339.10 CAD**
*   **Tariff Rate: 10%** -> New Price: **$126,489.00 CAD**
*   **Tariff Rate: 11%** -> New Price: **$127,638.90 CAD**
*   **Tariff Rate: 12%** -> New Price: **$128,788.80 CAD**
*   **Tariff Rate: 13%** -> New Price: **$129,938.70 CAD**
*   **Tariff Rate: 14%** -> New Price: **$131,088.60 CAD**
*   **Tariff Rate: 15%** -> New Price: **$132,238.50 CAD**
*   **Tariff Rate: 16%** -> New Price: **$133,388.40 CAD**
*   **Tariff Rate: 17%** -> New Price: **$134,538.30 CAD**
*   **Tariff Rate: 18%** -> New Price: **$135,688.20 CAD**
*   **Tariff Rate: 19%** -> New Price: **$136,838.10 CAD**
*   **Tariff Rate: 20%** -> New Price: **$137,988.00 CAD**

As the import tariff increases from 5% to 20%, the price of the Tesla Cybertruck in Canada would rise from $120,739.50 CAD to $137,988.00 CAD, assuming the tariff is directly applied to the base price and passed entirely to the consumer. This represents an increase of $17,248.50 CAD due to the tariff hike.